# Symbolic student — input NOISE robustness at 2 bits

Duplicate of `train_digitization_ablation.ipynb` with **the noise turned on**. Same student, same
three-phase schedule, same run-directory convention and the same parquet schema, so every arm
produced here drops straight into the digitization comparison cells of
`compare_baselines_vs_symbolic.ipynb` (`digi_*_nexp*` is the glob those cells discover).

Only the noise knob differs. `NOISE_SIGMA = SIGMA_NOISE_E = 80 e-` is the default here.

## The physics of the ordering (fixed, do not reorder)

`Q_in = Q_sim + eps`, `eps ~ N(0, sigma_noise)`, `sigma_noise = 80 e-` — arXiv:2602.15946 Eq. 6,
Sec. IV B (Cadence Virtuoso design simulated with Spectre). The noise is added to the **analog**
charge *by the generator*, and `DigitizeLayer` quantizes what comes out. That is the real chain: the
sensor and the preamp are noisy, the ADC digitizes the noisy charge. Because the generator runs with
`digitize=False`, the graph order is already correct — nothing to arrange.

`NOISE = -1` means OFF. Verified, not assumed: `OptimizedDataGenerator_v3._read_tfrecord` guards the
only noise site with `if self.noise != -1:`, and that site sits *above* its quantize/digitize
branches.

## What to watch, in this order

1. **Sign accuracy.** The gate reads `Tx`, `Ty` — the between-slice centroid *drift*. It is a small
   differential quantity built from two nearly equal numbers, so it is the most noise-fragile thing
   in the model and it will degrade **before** the position resolutions do. Cell 4 prints the
   logistic-regression ceiling under this arm's noise + thresholds, cell 9 prints what the trained
   gate achieves. Clean 2-bit reference: bal-acc 0.97 / 0.97; raw-pixel ceiling 0.966.
2. **T0 vs the noise.** With `sigma = 80 e-` and the published `T0 = 248 e-` = 3.1 sigma, a *pedestal*
   pixel fires with probability ~1e-3, i.e. ~0.25 hot pixels per 16x16x2 frame. Cell 3b measures this
   instead of trusting the arithmetic. This is the number that kills the superseded `T0 = 100 e-`
   placement (1.25 sigma -> ~10% of empty pixels firing).
3. **I68**, per output, never residual std.

## Arms

| # | `N_BITS` | `NOISE_SIGMA` | `TEST_NOISE_SIGMA` | out dir | purpose |
|---|---|---|---|---|---|
| 1 | 2 | **80** | None | `digi_2bit_paper_code_noise80_nexp1` | nominal — the arm that must work |
| 2 | 2 | 40 | None | `..._noise40_nexp1` | low |
| 3 | 2 | 160 | None | `..._noise160_nexp1` | high, 2x nominal |
| 4 | 2 | 0 | None | `digi_2bit_paper_code_nexp1` | clean 2-bit — **already trained** by the digitization notebook; do not redo it |
| 5 | None | 80 | None | `digi_analog_noise80_nexp1` | separates "noise hurts" from "noise + quantization hurt" |
| 6 | None | 0 | None | `digi_analog_nexp1` | clean analog — **already trained** |
| 7 | 2 | 0 | **80** | `digi_2bit_paper_code_test80_nexp1` | train clean / test noisy: separates *robust* from *trained-on-noise* |
| 8 | 2 | 80 | 0 | `..._noise80_test0_nexp1` | the reverse, to see what noise training costs on clean input |
| 9 | 2 | 80 | None, `TRAINABLE_THR=True` | `..._trainthr_noise80_nexp1` | does the ADC want higher thresholds when the input is noisy? |
| 10 | 2 | 80 | None, `N_EXPERTS=4` | `..._noise80_nexp4` | noise robustness of the MoE |

Arm 1 is the deliverable: **2-bit input AND nominal noise, simultaneously**. Arms 2/3 bracket it,
arms 5–8 attribute the damage, arm 9 asks whether the optimal ADC moves under noise, arm 10 checks
the MoE. Analog arms are reference points — the on-sensor ADC is 2-bit and no result may be reported
as "use N bits" for N != 2.

## How to use

Edit **cell 2 only** (`NOISE_SIGMA`, and `TEST_NOISE_SIGMA` for the mismatch arms), then
Restart & Run All. The run directory name and `summary.json["noise"]` both record the train and test
sigma, so an arm can never be mistaken for another one later.

## Reporting

Cell 9 prints **I68** (the paper metric: half-width of the minimal 68% interval), Gaussian-fit pull
mean and sigma, and sign accuracy for alpha and beta — all four outputs, physical units. Never quote
residual std: it is tail-driven and produced a spurious non-monotonicity in an earlier version of the
bit-depth table. The `i68` and pull-fit definitions are copied verbatim from
`compare_baselines_vs_symbolic.ipynb`, and `summary.json["noise"]` carries both sigmas, so the noise
grid drops straight into section B of that notebook (`noise_tr` / `noise_te` columns).


In [1]:
# ---- 1. setup ----
import os
WORKDIR = "/depot/cms/private/users/kuang14/Smart_Pixel/smart-pixels-digitization"
HELPERS = os.path.join(WORKDIR, "two_bit_optimization_helpers")
os.chdir(WORKDIR)

import sys, json, glob, time, math
sys.path.insert(0, HELPERS)

import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_probability as tfp

for g in tf.config.list_physical_devices("GPU"):
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception: pass

from prepare_tfrecords import generate_tfrecords, load_tfrecords
from loss import custom_loss                       # original loss: used only for the init smoke-test
from models.student_max import build_student_max, pack_14

# the TimeGrad ansatz (self-signed) must be the one on disk
assert "sign_k_alpha" in open(os.path.join(HELPERS, "symbolic", "ansatz.py")).read(), \
       "TimeGrad ansatz NOT on disk -- run cell 1 of distill_moe_timegrad_sign.ipynb, then restart kernel"
print("TF", tf.__version__, "| TimeGrad ansatz on disk: OK | GPUs:", tf.config.list_physical_devices("GPU"))
pi = np.pi

2026-08-11 19:09:47.275427: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-08-11 19:09:47.275515: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-08-11 19:09:47.276621: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-11 19:09:47.284417: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-08-11 19:09:49.135547: W tensorflow/compiler/tf2

TF 2.15.1 | TimeGrad ansatz on disk: OK | GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# ---- 2. config ----   *** set the NOISE ARM here, then Restart & Run All ***
SEED = 42
tf.random.set_seed(SEED); np.random.seed(SEED)

DATASET_DIR      = "/depot/cms/users/kuang14/Smart_Pixel/dataset_s_series/dataset_3sr/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets"
SELECT_CONTAINED = True
TIMESLICES       = 2
BATCH            = 5000
TFRECORDS_EXIST  = True

VARIANT   = "barycenter"

# Published constants live in symbolic/digitize.py -- never retype them here.
from symbolic.digitize import (SIGMA_NOISE_E, FIVE_SIGMA_E,
                               PAPER_THRESHOLDS_2BIT, PAPER_REF)

# ======== DIGITIZATION (paper Sec. IV) ========
N_BITS      = 2          # 2 IS THE DEPLOYMENT POINT. None/4/3 are reference only.
THR_SCHEME  = "paper"    # 'paper'    -> published (248, 668, 1663) e-, 2-bit ONLY
                         # 'lowfirst' -> occupancy-derived with T0 floored at the
                         #               published 248 e-. Required for 3/4-bit,
                         #               where no published values exist.
LEVEL_MODE  = "code"     # 'code'     -> bin index 0..3. Paper Eq. 1, and the ONLY
                         #               apples-to-apples config vs the baselines.
                         # 'midpoint' -> bin-centre charge; assumes a 4-entry
                         #               code->charge LUT on-chip that the
                         #               baselines do not have. Report any credit
                         #               it earns as bought with chip area.
OFFSET      = SIGMA_NOISE_E   # 80 e- = sigma_noise (Cadence Virtuoso / Spectre)
STE         = True            # hard forward always; STE only on the backward pass

# ---- Fix 2: trainable thresholds (paper Sec. IV A) ----
TRAINABLE_THR   = False       # True -> co-train T with the physics
T_MIN           = 240.0       # Eq. 3 lower bound; the learned T0 cannot go below
                              # this, which is what keeps it out of the noise
PARAMETRIZATION = "paper"     # Delta = softplus(theta), Eq. 2. 'legacy' is the
                              # old softplus(expm1(theta)) form -- same values,
                              # ~200x larger threshold gradients. Do not use it
                              # when thresholds are trainable.
# k is annealed only when the thresholds need gradient. A large fixed k makes the
# sigmoids step-like, which is right for a frozen quantizer and useless for a
# trainable one.
K_INIT, K_MAX = (1.0, 67.0) if TRAINABLE_THR else (50.0, 50.0)

# ---- Fix 3: input noise. Q_in = Q_sim + eps,  eps ~ N(mu, sigma)  (Eq. 6) ----
# Order is physical and fixed: the generator adds noise to the ANALOG charge, then
# DigitizeLayer quantizes. Verified in OptimizedDataGenerator_v3._read_tfrecord.
NOISE_MU         = 40
NOISE_SIGMA      = SIGMA_NOISE_E   # <<< THE knob of this notebook. 80 e- = nominal
                              # (Sec. IV B, Cadence Virtuoso / Spectre).
                              # grid: 0 (clean) / 40 (low) / 80 (nominal) / 160 (high)
TEST_NOISE_SIGMA = None       # None = evaluate at the training noise.
                              # Mismatch arms, which separate "the model is robust"
                              # from "the model was trained on noise":
                              #   NOISE_SIGMA=0,  TEST_NOISE_SIGMA=80  train clean, test noisy
                              #   NOISE_SIGMA=80, TEST_NOISE_SIGMA=0   train noisy, test clean
assert NOISE_SIGMA > 0 or TEST_NOISE_SIGMA, \
    "both noise knobs are off -- that arm belongs to train_digitization_ablation.ipynb"

N_EXPERTS = 1                    # 1 (single formula), 2, or 4
ROUTER_HIDDEN = {4: (32, 16), 2: (16,)}.get(N_EXPERTS, ())
ROUTER_TEMP   = 1.0
EXPERT_KWARGS = {}

DIGITIZE = False   # generator must NOT digitize: DigitizeLayer is the only quantizer

# training schedule
LR        = 1e-3
EPOCHS_P1 = 150       # phase 1: means (MSE)
EPOCHS_P2 = 200       # phase 2: means+cov refine (original NLL)
EPOCHS_P3 = 120       # phase 3: error head only (corrected NLL)

# ---- run tag: every knob that changes the answer appears in the directory name ----
_bits = "analog" if N_BITS is None else f"{N_BITS}bit"
_tag  = _bits if N_BITS is None else f"{_bits}_{THR_SCHEME}_{LEVEL_MODE}"
if TRAINABLE_THR:            _tag += "_trainthr"
if NOISE_SIGMA > 0:          _tag += f"_noise{int(NOISE_SIGMA)}"
if TEST_NOISE_SIGMA is not None: _tag += f"_test{int(TEST_NOISE_SIGMA)}"
OUT_DIR = f"/depot/cms/private/users/kuang14/Smart_Pixel/digi_{_tag}_nexp{N_EXPERTS}"
os.makedirs(OUT_DIR, exist_ok=True)

assert THR_SCHEME != "paper" or N_BITS in (2, None), \
    "scheme='paper' publishes two-bit thresholds only -- use 'lowfirst' for 3/4-bit"
assert not (TRAINABLE_THR and N_BITS is None), "nothing to train in analog mode"

print(f"N_EXPERTS={N_EXPERTS} | ROUTER_HIDDEN={ROUTER_HIDDEN}")
print(f"digitization : {_tag}")
print(f"  thresholds : scheme={THR_SCHEME}  level_mode={LEVEL_MODE}  offset={OFFSET} e-")
print(f"  trainable  : {TRAINABLE_THR}  T_min={T_MIN if TRAINABLE_THR else OFFSET} e-  "
      f"param={PARAMETRIZATION}  k: {K_INIT} -> {K_MAX}")
print(f"  noise      : train sigma={NOISE_SIGMA} e-  test sigma="
      f"{NOISE_SIGMA if TEST_NOISE_SIGMA is None else TEST_NOISE_SIGMA} e-")
print(f"OUT_DIR: {OUT_DIR}")


N_EXPERTS=1 | ROUTER_HIDDEN=()
digitization : 2bit_paper_code_noise80
  thresholds : scheme=paper  level_mode=code  offset=80.0 e-
  trainable  : False  T_min=80.0 e-  param=paper  k: 50.0 -> 50.0
  noise      : train sigma=80.0 e-  test sigma=80.0 e-
OUT_DIR: /depot/cms/private/users/kuang14/Smart_Pixel/digi_2bit_paper_code_noise80_nexp1


In [3]:
# ---- 3. data + ADC thresholds ----
# Noise is added to the ANALOG charge by the generator; DigitizeLayer quantizes
# afterwards. NOISE = -1 means OFF (verified: OptimizedDataGenerator_v3 line ~711,
# `if self.noise != -1:` guards the only place noise is applied, and it sits above
# the quantize/digitize branches).
NOISE = -1 if NOISE_SIGMA <= 0 else [float(NOISE_MU), float(NOISE_SIGMA)]

_, _, tfr_tr, tfr_val = generate_tfrecords(
    dataset_dir=DATASET_DIR, model_type="ViT_Max",     # tfrecord format only; NO teacher is loaded
    train_batch_size=BATCH, val_batch_size=BATCH,
    select_contained=SELECT_CONTAINED, timeslices=TIMESLICES,
    tfrecords_exist=TFRECORDS_EXIST, seed=SEED,
)
tg, vg = load_tfrecords(tfr_tr, tfr_val, noise=NOISE, digitize=DIGITIZE, seed=SEED)

# train-clean / test-noisy cross-arm: a SECOND validation generator at a different
# noise level. Everything downstream evaluates EVAL_GEN, not vg.
if TEST_NOISE_SIGMA is None:
    EVAL_GEN, EVAL_NOISE = vg, NOISE_SIGMA
else:
    _test_noise = -1 if TEST_NOISE_SIGMA <= 0 else [float(NOISE_MU), float(TEST_NOISE_SIGMA)]
    _, EVAL_GEN = load_tfrecords(tfr_tr, tfr_val, noise=_test_noise,
                                 digitize=DIGITIZE, seed=SEED)
    EVAL_NOISE = TEST_NOISE_SIGMA
    print(f"cross-arm: trained at sigma={NOISE_SIGMA} e-, evaluated at sigma={EVAL_NOISE} e-")

labels_scale = json.load(open(os.path.join(tfr_tr, "metadata.json")))["labels_scale"]
print("labels_scale:", labels_scale, "| train batches:", len(tg), "| val batches:", len(vg))

xb, yb = tg[0]; xb = np.asarray(xb); yb = np.asarray(yb)
assert xb.shape[1:] == (16, 16, 2) and yb.shape[1] == 4
assert DIGITIZE is False, "generator digitize=True would quantize twice -- set DIGITIZE=False"
print("x:", xb.shape, "| y:", yb.shape, "| charge range:", float(xb.min()), float(xb.max()))

# ---- ADC thresholds ----
from symbolic.digitize import build_thresholds, levels_for

if N_BITS is None:
    THRESHOLDS = LEVELS = BIN_EDGES = None
    THR_PROV = dict(n_bits=None, source="analog passthrough -- no quantizer")
    print("analog inputs -- DigitizeLayer is a passthrough")
else:
    # Derived schemes see the charge the model will see, noise included (the paper
    # optimises its thresholds on noisy input too). scheme='paper' ignores this.
    _Xthr = np.concatenate([np.asarray(tg[i][0], "float32") for i in range(min(4, len(tg)))])
    THRESHOLDS, BIN_EDGES, THR_PROV = build_thresholds(
        _Xthr, N_BITS, scheme=THR_SCHEME, offset=OFFSET)
    LEVELS = levels_for(N_BITS, LEVEL_MODE, BIN_EDGES)
    _occ = np.bincount(np.digitize(_Xthr[_Xthr > OFFSET], THRESHOLDS),
                       minlength=2 ** N_BITS) / max((_Xthr > OFFSET).sum(), 1)
    print(f"thresholds (e-): {np.round(THRESHOLDS, 1)}")
    print(f"  source       : {THR_PROV['source']}")
    print(f"  T0 = {THRESHOLDS[0]:.1f} e- = {THR_PROV['t0_over_sigma_noise']:.2f} sigma_noise"
          f"  (5 sigma = {FIVE_SIGMA_E:.0f} e-, below it: {THR_PROV['t0_below_five_sigma']})")
    if THR_PROV.get("published_thresholds_e"):
        _d = np.asarray(THRESHOLDS) - np.asarray(THR_PROV["published_thresholds_e"])
        print(f"  vs published : {np.round(_d, 1)} e-")
    for _n in THR_PROV.get("notes", []):
        print(f"  NOTE: {_n}")
    print(f"levels ({LEVEL_MODE}): {np.round(LEVELS, 2)}")
    print(f"occupancy/code (above-offset pixels): {np.round(_occ, 3)}")
    THR_PROV["occupancy_per_code"] = [float(v) for v in _occ]
    del _Xthr

def digitize_np(x):
    """numpy mirror of DigitizeLayer's hard path, for the cell-4 sign calibration."""
    x = np.asarray(x, "float32")
    if N_BITS is None:
        return x
    return LEVELS[np.clip(np.digitize(x, THRESHOLDS), 0, len(LEVELS) - 1)].astype("float32")


Loading metadata from /depot/cms/users/kuang14/Smart_Pixel/dataset_s_series/dataset_3sr/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets/TFR_files/2t/TFR_train_contained/metadata.json
Loading metadata from /depot/cms/users/kuang14/Smart_Pixel/dataset_s_series/dataset_3sr/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets/TFR_files/2t/TFR_test_contained/metadata.json


Loading metadata from /depot/cms/users/kuang14/Smart_Pixel/dataset_s_series/dataset_3sr/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets/TFR_files/2t/TFR_train_contained/metadata.json
Loading metadata from /depot/cms/users/kuang14/Smart_Pixel/dataset_s_series/dataset_3sr/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets/TFR_files/2t/TFR_test_contained/metadata.json


2026-08-11 19:09:54.537916: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2574 MB memory:  -> device: 0, name: NVIDIA A100-PCIE-40GB MIG 1g.5gb, pci bus id: 0000:81:00.0, compute capability: 8.0


labels_scale: [123.41016201181557, 30.929849811025303, 6.577498885094723, 1.9295648020239338] | train batches: 87 | val batches: 22
x: (5000, 16, 16, 2) | y: (5000, 4) | charge range: -3368.926513671875 38461.296875
thresholds (e-): [ 248.  668. 1663.]
  source       : published verbatim, arXiv:2602.15946 Sec. IV / Fig. 3 (Max transformer, SoftQuantize)
  T0 = 248.0 e- = 3.10 sigma_noise  (5 sigma = 400 e-, below it: True)
  vs published : [0. 0. 0.] e-
levels (code): [0. 1. 2. 3.]
occupancy/code (above-offset pixels): [0.865 0.059 0.03  0.046]


In [4]:
# ---- 3b. what the noise actually does to the input (measured, not assumed) ----
# Three numbers decide whether an arm is sane before a single epoch runs.
#
# (a) the generator really is applying the sigma we asked for. Negative pixel charge can
#     ONLY come from the noise, so the negative tail is a clean half-normal estimator of
#     sigma -- no assumption about event ordering or about the generator's internals.
# (b) the pure-noise firing rate: how often a PEDESTAL pixel crosses T0. This is what
#     "T0 sits in the noise" means quantitatively -- 248 e- is 3.1 sigma, the superseded
#     100 e- is 1.25 sigma.
# (c) the code-flip rate: how many pixels land in a different 2-bit code than they would
#     have on clean charge. That is the input perturbation the physics moments must absorb.
#     Measured on the clean charge with a numpy replica of Eq. 6 (Q_in = Q_sim + eps), so
#     it needs no event-by-event alignment between two generators.
Xn = np.concatenate([np.asarray(tg[i][0], "float32") for i in range(min(4, len(tg)))])

neg = Xn[Xn < 0.0]
sig_est = float(np.sqrt(np.mean(neg ** 2))) if neg.size else 0.0
print(f"generator noise: requested sigma = {NOISE_SIGMA:.0f} e-  |  measured from the "
      f"negative tail = {sig_est:.0f} e-  ({neg.size / Xn.size:.1%} of pixels < 0)")
if NOISE_SIGMA > 0:
    assert 0.6 * NOISE_SIGMA < sig_est < 1.4 * NOISE_SIGMA, (
        f"generator noise sigma looks like {sig_est:.0f} e-, not {NOISE_SIGMA:.0f} -- "
        "check NOISE plumbing in cell 3 before training")
else:
    assert neg.size == 0, "noise is supposed to be OFF but negative charge is present"

if N_BITS is None:
    print("analog arm -- no codes to flip; the noise enters the charge-weighted moments directly")
else:
    _tg_clean, _ = load_tfrecords(tfr_tr, tfr_val, noise=-1, digitize=DIGITIZE, seed=SEED)
    Xq = np.concatenate([np.asarray(_tg_clean[i][0], "float32")
                         for i in range(min(4, len(_tg_clean)))])
    rng = np.random.default_rng(SEED)
    Xr = Xq + rng.normal(NOISE_MU, max(NOISE_SIGMA, 0.0), Xq.shape).astype("float32")

    code = lambda a: np.digitize(a, THRESHOLDS)      # 0..2^n-1, the layer's own edges
    empty = Xq <= 0.0
    rate = float(((code(Xr) > 0) & empty).sum() / max(empty.sum(), 1))
    analytic = (0.0 if NOISE_SIGMA <= 0 else
                0.5 * math.erfc((THRESHOLDS[0] - NOISE_MU) / (NOISE_SIGMA * np.sqrt(2.0))))
    per_frame = rate * float(empty.mean()) * 16 * 16 * TIMESLICES
    flip = code(Xr) != code(Xq)
    sig_px = Xq > THRESHOLDS[0]

    print(f"\nT0 = {THRESHOLDS[0]:.0f} e- = "
          f"{THRESHOLDS[0] / max(NOISE_SIGMA, 1e-9):.2f} sigma_noise")
    print(f"  pedestal pixels firing  : {rate:.2e} measured | {analytic:.2e} analytic "
          f"(0.5*erfc(T0/sigma/sqrt2))")
    print(f"  hot pixels per frame    : {per_frame:.3f}  (16x16x{TIMESLICES})")
    print(f"  code flips, all pixels  : {flip.mean():.3%}")
    print(f"  code flips, signal only : {flip[sig_px].mean():.3%}  "
          f"({sig_px.mean():.2%} of pixels are above T0 when clean)")
    if rate > 0.01:
        print("  WARNING: >1% of empty pixels fire -- every cluster reads wider than it is and "
              "the centroid drift the sign gate needs is buried. Raise T0.")
    del Xq, Xr, _tg_clean
del Xn


generator noise: requested sigma = 80 e-  |  measured from the negative tail = 84 e-  (29.4% of pixels < 0)
Loading metadata from /depot/cms/users/kuang14/Smart_Pixel/dataset_s_series/dataset_3sr/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets/TFR_files/2t/TFR_train_contained/metadata.json
Loading metadata from /depot/cms/users/kuang14/Smart_Pixel/dataset_s_series/dataset_3sr/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets/TFR_files/2t/TFR_test_contained/metadata.json



T0 = 248 e- = 3.10 sigma_noise
  pedestal pixels firing  : 4.58e-03 measured | 4.66e-03 analytic (0.5*erfc(T0/sigma/sqrt2))
  hot pixels per frame    : 2.113  (16x16x2)
  code flips, all pixels  : 1.321%
  code flips, signal only : 6.581%  (3.69% of pixels are above T0 when clean)


In [5]:
# ---- 4. calibrate the TimeGrad sign gates from TRAIN data (exact k, a0 inits) ----
# 1-feature logistic  P(sign=+) = sigma(w*T + b)  maps to  tanh(k*(a0 - T)),  k = -w/2, a0 = -b/w.
# alpha sign <- Tx (x-drift: COLS x 50 um); beta sign <- Ty (y-drift: ROWS x 12.5 um).
# Axis convention measured in axis_convention_diagnostic.ipynb; matches ansatz.py v3. Fit TRAIN, verify VAL.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score

def drifts_np(X):                               # X (N,16,16,2) -> Tx, Ty in um
    q = np.maximum(np.asarray(X, "float32"), 0.0)
    row, col = np.meshgrid(np.arange(16), np.arange(16), indexing="ij")
    xw = (col * 50.0)[None]; yw = (row * 12.5)[None]   # x <- cols @ 50 um, y <- rows @ 12.5 um
    q0 = q[..., 0].sum((1, 2)) + 1e-6; q1 = q[..., 1].sum((1, 2)) + 1e-6
    Tx = (q[..., 1] * xw[0]).sum((1, 2)) / q1 - (q[..., 0] * xw[0]).sum((1, 2)) / q0
    Ty = (q[..., 1] * yw[0]).sum((1, 2)) / q1 - (q[..., 0] * yw[0]).sum((1, 2)) / q0
    return Tx, Ty

def collect(gen, n_batches=None):
    n = len(gen) if n_batches is None else min(n_batches, len(gen))
    Xs, Ys = zip(*[(np.asarray(gen[i][0]), np.asarray(gen[i][1])) for i in range(n)])
    return np.concatenate(Xs), np.concatenate(Ys)

Xtr_, Ytr_ = collect(tg, n_batches=8)
Xva_, Yva_ = collect(vg)
# calibrate on what the model will actually see, not on analog
Tx_tr, Ty_tr = drifts_np(digitize_np(Xtr_)); Tx_va, Ty_va = drifts_np(digitize_np(Xva_))

SIGN_INIT = {}
for nm, T_tr, T_va, i in [("alpha", Tx_tr, Tx_va, 2), ("beta", Ty_tr, Ty_va, 3)]:
    ys_tr = (Ytr_[:, i] > 0).astype(int)
    clf = LogisticRegression(max_iter=5000).fit(T_tr.reshape(-1, 1), ys_tr)
    w, b = float(clf.coef_[0, 0]), float(clf.intercept_[0])
    k, a0 = -w / 2.0, -b / w
    # evaluate the gate EXACTLY as the ansatz does: sign(tanh(k*(a0-T))) = sign(k*(a0-T)).
    # Dropping k here hid a convention flip whenever the logistic fit returned k < 0.
    hard = np.where(k * (a0 - T_va) > 0, 1.0, -1.0)
    agr  = (hard == np.sign(Yva_[:, i])).mean()
    bal  = balanced_accuracy_score((Yva_[:, i] > 0).astype(int), (hard > 0).astype(int))
    SIGN_INIT[f"initial_sign_k_{nm}"]  = k
    SIGN_INIT[f"initial_sign_a0_{nm}"] = a0
    print(f"cot{nm}: k={k:+.4f}/um  a0={a0:+.3f} um  | VAL raw agree {agr:.4f}  bal-acc {bal:.4f}")
    # The published thresholds are new here, so there is no prior measurement of
    # the gate's accuracy under them -- print it and only stop on a real failure.
    if bal < 0.90:
        print(f"  NOTE: cot{nm} gate is at {bal:.4f} under {_tag}. The old occupancy "
              f"thresholds gave 0.9689/0.9484 at 2-bit; this arm has no prior value.")
    assert bal > 0.70, (f"cot{nm} sign calibration {bal:.4f} at {_tag} -- the gate is "
                        "not learning the sign at all. Check thresholds/level_mode.")
print("\nSIGN_INIT:", {k: round(v, 4) for k, v in SIGN_INIT.items()})
json.dump(SIGN_INIT, open(os.path.join(OUT_DIR, "sign_init.json"), "w"), indent=1)
del Xtr_, Ytr_

cotalpha: k=+0.0235/um  a0=-0.099 um  | VAL raw agree 0.8899  bal-acc 0.8899
  NOTE: cotalpha gate is at 0.8899 under 2bit_paper_code_noise80. The old occupancy thresholds gave 0.9689/0.9484 at 2-bit; this arm has no prior value.
cotbeta: k=+0.1033/um  a0=-10.922 um  | VAL raw agree 0.8857  bal-acc 0.8771
  NOTE: cotbeta gate is at 0.8771 under 2bit_paper_code_noise80. The old occupancy thresholds gave 0.9689/0.9484 at 2-bit; this arm has no prior value.

SIGN_INIT: {'initial_sign_k_alpha': 0.0235, 'initial_sign_a0_alpha': -0.0994, 'initial_sign_k_beta': 0.1033, 'initial_sign_a0_beta': -10.9221}


In [6]:
# ---- 5. student model  (DigitizeLayer -> router + experts) ----
from symbolic.digitize import DigitizeLayer

def make_digitizer():
    # T_min is the floor the cumulative-sum construction starts from (Eq. 3), so
    # when thresholds are trainable it is what keeps the learned T0 out of the noise.
    return DigitizeLayer(n_bits=N_BITS, thresholds=THRESHOLDS, levels=LEVELS,
                         threshold_offset=(T_MIN if TRAINABLE_THR else OFFSET),
                         ste=STE,
                         trainable_thresholds=TRAINABLE_THR, trainable_levels=False,
                         initial_k=K_INIT, parametrization=PARAMETRIZATION,
                         name="digitize")

class RouterFeatures(tf.keras.layers.Layer):
    # HARDENED: relu charge, std floored, finite-guard + clip.
    def call(self, x):
        xp  = tf.nn.relu(x)
        q   = tf.reduce_sum(xp, axis=-1)
        px  = tf.reduce_sum(q, axis=2)
        py  = tf.reduce_sum(q, axis=1)
        tot = tf.reduce_sum(py, axis=1, keepdims=True) + 1e-9
        pxn, pyn = px / tot, py / tot
        yc  = tf.range(16, dtype=tf.float32) - 7.5
        mu  = tf.reduce_sum(pyn * yc, axis=1, keepdims=True)
        d   = yc[None, :] - mu
        m2  = tf.reduce_sum(pyn * d**2, axis=1, keepdims=True)
        m3  = tf.reduce_sum(pyn * d**3, axis=1, keepdims=True)
        std = tf.sqrt(m2 + 1.0)
        skew = m3 / (std ** 3)
        py0 = tf.reduce_sum(xp[..., 0],  axis=1)
        py1 = tf.reduce_sum(xp[..., -1], axis=1)
        c0  = tf.reduce_sum(py0*yc, 1, keepdims=True) / (tf.reduce_sum(py0, 1, keepdims=True) + 1e-9)
        c1  = tf.reduce_sum(py1*yc, 1, keepdims=True) / (tf.reduce_sum(py1, 1, keepdims=True) + 1e-9)
        tasym = c1 - c0
        logq  = tf.math.log(tot + 1.0)
        f = tf.concat([pxn, pyn, std, skew, tasym, logq], axis=-1)
        f = tf.where(tf.math.is_finite(f), f, tf.zeros_like(f))
        return tf.clip_by_value(f, -8.0, 8.0)

# The old AxisFixExpert wrapper is GONE. The transpose it patched is now fixed
# inside ansatz.py (v3), for the pitches, widths and sign observables as well as
# the two position slots. Keeping the wrapper would DOUBLE-swap and reproduce the
# original bug exactly (x-slot corr(true_x) ~ 0.00, corr(true_y) ~ 0.92).
def make_expert():
    return build_student_max(VARIANT,
                             ansatz_kwargs={"labels_scale": labels_scale, **SIGN_INIT},
                             **EXPERT_KWARGS)

class MoECore(tf.keras.Model):
    def __init__(self, n_experts, router_hidden, temp, **kw):
        super().__init__(**kw)
        self.n_experts, self.temp = n_experts, temp
        self.digi   = make_digitizer()
        self.feats  = RouterFeatures(name="router_features")
        self.router = tf.keras.Sequential(
            [tf.keras.layers.Dense(h, activation="relu") for h in router_hidden]
            + [tf.keras.layers.Dense(n_experts)], name="router")
        self.experts = [make_expert() for _ in range(n_experts)]
    def _route_d(self, xd):                       # xd is ALREADY digitized
        return tf.nn.softmax(self.router(self.feats(xd)) / self.temp, axis=-1)
    def route(self, x):                           # public: takes raw input
        return self._route_d(self.digi(x, training=False))
    def call(self, x, training=False):
        xd = self.digi(x, training=training)      # digitize ONCE, share with router+experts
        w = self._route_d(xd)
        outs  = [e(xd, training=training) for e in self.experts]
        means = tf.add_n([w[:, k:k+1] * outs[k][0] for k in range(self.n_experts)])
        chol  = tf.add_n([w[:, k:k+1] * outs[k][1] for k in range(self.n_experts)])
        return means, chol

class SingleCore(tf.keras.Model):                 # N_EXPERTS == 1: no router at all
    def __init__(self, **kw):
        super().__init__(**kw)
        self.digi = make_digitizer()
        self.experts = [make_expert()]
    def call(self, x, training=False):
        return self.experts[0](self.digi(x, training=training), training=training)

class PackedStudent(tf.keras.Model):              # (mu, chol) -> packed 14 for the loss
    def __init__(self, core, **kw):
        super().__init__(**kw)
        self.core = core
    def call(self, x, training=False):
        mu, chol = self.core(x, training=training)
        return pack_14(mu, chol)

core  = SingleCore(name="single_core") if N_EXPERTS == 1 else \
        MoECore(N_EXPERTS, ROUTER_HIDDEN, ROUTER_TEMP, name="moe_core")
model = PackedStudent(core, name=f"symbolic_n{N_EXPERTS}")
assert model(xb[:64]).shape[-1] == 14

n_router = core.router.count_params() if N_EXPERTS > 1 else 0
print(f"total params: {model.count_params()} | router: {n_router} "
      f"| per-expert: {core.experts[0].count_params()}")

# TimeGrad sign scalars must be live in EVERY expert
names = [v.name for v in model.trainable_variables]
for need in ("sign_k_alpha", "sign_a0_alpha", "sign_k_beta", "sign_a0_beta"):
    n_found = sum(need in n for n in names)
    assert n_found == N_EXPERTS, f"{need}: found {n_found}, expected {N_EXPERTS} -- stale ansatz.py / kernel not restarted"
print("TimeGrad sign scalars present in all", N_EXPERTS, "expert(s)")

# ---- axis convention: probe the LOADED class, not the file on disk ----
# inspect.getsource() re-reads ansatz.py, so it would pass even with stale bytecode
# or a stale copy of THIS notebook. Light one pixel at (row 0, col 15): x is the
# column axis at 50 um, so the x slot must read (15-7.5)*50 = +375 um.
from symbolic import ansatz as _ans
from symbolic import PhysicsAnsatz as _PA
_z = np.zeros((1, 16, 16, TIMESLICES), "float32"); _z[0, 0, 15, :] = 1000.0
_o = np.asarray(_PA(variant="barycenter", labels_scale=[1.0, 1.0, 1.0, 1.0])(tf.constant(_z)))[0]
print(f"  ansatz module: {_ans.__file__}")
print(f"  axis probe: x-slot={_o[0]:+8.1f} um   y-slot={_o[1]:+8.1f} um   (expect x=+375.0)")
assert _o[0] > 300.0, (
    f"AXIS FIX NOT LIVE: x-slot={_o[0]:+.1f}, expected +375. Either the kernel holds a "
    "stale ansatz.py (Kernel > Restart), or this notebook still wraps the expert in "
    "AxisFixExpert (File > Reload Notebook from Disk).")
print("axis convention: OK (x on columns @ 50 um)")

# digitizer is live and sits in front of everything
_probe_in = np.asarray(xb[:256], "float32")
_probe_out = np.asarray(core.digi(tf.constant(_probe_in), training=False))
if N_BITS is None:
    assert np.allclose(_probe_in, _probe_out), "analog mode must be a passthrough"
    print("digitizer: passthrough (analog)")
else:
    _u = np.unique(_probe_out)
    assert len(_u) <= 2 ** N_BITS, f"{len(_u)} distinct outputs, expected <= {2 ** N_BITS}"
    print(f"digitizer: {N_BITS}-bit live | distinct outputs {len(_u)}: {np.round(_u, 2)}")

2026-08-11 19:10:10.220964: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907


total params: 376 | router: 0 | per-expert: 372
TimeGrad sign scalars present in all 1 expert(s)
  ansatz module: /depot/cms/private/users/kuang14/Smart_Pixel/smart-pixels-digitization/two_bit_optimization_helpers/symbolic/ansatz.py
  axis probe: x-slot=  +375.0 um   y-slot=   -73.5 um   (expect x=+375.0)
axis convention: OK (x on columns @ 50 um)
digitizer: 2-bit live | distinct outputs 4: [0. 1. 2. 3.]


In [7]:
# ---- 6. pre-train checks: axis wiring + init sign agreement + NaN scan ----
cb, lb = np.asarray(vg[0][0]), np.asarray(vg[0][1])
mu0 = np.asarray(core(cb, training=False)[0])

# (a) axis fix verified: x-slot tracks true_x, y-slot tracks true_y (both ~0.92)
for nm, i in [("x", 0), ("y", 1)]:
    cx = np.corrcoef(mu0[:, i], lb[:, 0])[0, 1]; cy = np.corrcoef(mu0[:, i], lb[:, 1])[0, 1]
    print(f"  {nm}-slot: corr(true_x)={cx:+.3f}  corr(true_y)={cy:+.3f}")
    assert abs([cx, cy][i]) > 0.8, f"{nm} not on its own axis -- axis fix broken, STOP"

# (b) init sign agreement (aff=(1,0) at init, so sign(cot slot) == ansatz TimeGrad sign)
for nm, i in [("cotA", 2), ("cotB", 3)]:
    m = mu0[:, i] != 0
    agr = (np.sign(mu0[m, i]) == np.sign(lb[m, i])).mean()
    print(f"  {nm}: init sign agreement {agr:.4f}  (on {m.mean():.1%} nonzero-|cot| events)")
    assert agr > 0.90, f"{nm} init sign broken -- do NOT train"

# (c) init loss finite + full-val NaN scan
l0 = float(tf.reduce_mean(custom_loss(tf.constant(yb, tf.float32), model(xb, training=False))))
print("  init custom_loss:", round(l0, 4)); assert np.isfinite(l0)
bad = sum(0 if np.isfinite(model(vg[i][0], training=False).numpy()).all() else 1 for i in range(len(vg)))
print("  NaN scan | bad val batches:", bad, "/", len(vg)); assert bad == 0
if N_EXPERTS > 1:
    print("  router usage (untrained, ~%.2f each):" % (1 / N_EXPERTS), core.route(xb).numpy().mean(0).round(3))
print("all pre-train checks passed")

  x-slot: corr(true_x)=+0.923  corr(true_y)=-0.001
  y-slot: corr(true_x)=+0.004  corr(true_y)=+0.912
  cotA: init sign agreement 0.8804  (on 99.8% nonzero-|cot| events)


AssertionError: cotA init sign broken -- do NOT train

In [ ]:
# ---- 7. train phase 1 (means, MSE) + phase 2 (means+cov, original NLL) ----
class NaNStop(tf.keras.callbacks.Callback):
    def on_train_batch_end(self, b, logs=None):
        v = (logs or {}).get("loss")
        if v is not None and not np.isfinite(v):
            print(f"\nNaN loss at batch {b} -- stopping"); self.model.stop_training = True

# Cosine k-annealing, paper Sec. IV A: k ~ 1 (soft, thresholds get gradient)
# -> k ~ 67 (hard). Only meaningful while the thresholds are trainable.
from symbolic.digitize import AnnealK
ANNEAL_CB = (AnnealK(core.digi, k_init=K_INIT, k_max=K_MAX, verbose=1)
             if (N_BITS is not None and TRAINABLE_THR) else None)
_anneal = [ANNEAL_CB] if ANNEAL_CB is not None else []

def freeze_adc_thresholds(q, model):
    """Freeze the learned ADC thresholds, whatever the Keras build.

    `Variable.trainable` is READ-ONLY in this training env (Keras 2.15 -> tf.Variable):
    assigning it raises "AttributeError: can't set attribute", while `.assign()` on the
    same variable works. Try the variable, then the sublayer, then the private flag, and
    verify against `model.trainable_variables` -- the only list the optimizer reads.
    Freezing the sublayer is equivalent here: threshold_deltas_raw is the
    SoftQuantizeLayer's only trainable weight (levels and k are non-trainable).
    """
    v = q.threshold_deltas_raw
    live = lambda: any(w is v for w in model.trainable_variables)
    for how, apply in (("variable", lambda: setattr(v, "trainable", False)),
                       ("sublayer", lambda: setattr(q, "trainable", False)),
                       ("private",  lambda: setattr(v, "_trainable", False))):
        try:
            apply()
        except AttributeError:
            continue
        if not live():
            try:
                v._trainable = False
            except Exception:
                pass
            return how
    raise RuntimeError(
        "ADC thresholds are STILL in model.trainable_variables -- they would keep "
        "moving while phases 2-3 fit against a shifting input distribution.")

def mse_means(y_true, y_pred14):
    mu = tf.gather(y_pred14, [0, 2, 4, 6], axis=-1)
    return tf.reduce_mean(tf.square(y_true - mu), axis=-1)

# phase 1: means only. Prevents the NLL sigma-inflation collapse (which kills the mean gradient).
model.compile(optimizer=tf.keras.optimizers.Adam(LR, clipnorm=1.0), loss=mse_means)
t0 = time.time()
h1 = model.fit(tg, validation_data=vg, epochs=EPOCHS_P1, shuffle=False, verbose=1,
               callbacks=_anneal + [tf.keras.callbacks.CSVLogger(os.path.join(OUT_DIR, "history_p1.csv")),
                          NaNStop(),
                          tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=30,
                                                           restore_best_weights=True, verbose=1)])
print("phase 1 done in %.0fs" % (time.time() - t0))

# The thresholds are learned HERE and nowhere else: phase 1 is the only stretch
# where k is soft enough for them to receive gradient. Freeze them at the end of
# it, so phases 2 and 3 fit the physics and the sigmas against a fixed ADC --
# exactly the quantizer the chip would ship.
THRESHOLDS_LEARNED = None
if N_BITS is not None:
    print("  thresholds after phase 1 (e-):", np.round(np.asarray(core.digi.thresholds), 1))
    if TRAINABLE_THR:
        core.digi.quantizer.log_k.assign([np.log(np.float32(K_MAX))])
        _how = freeze_adc_thresholds(core.digi.quantizer, model)
        THRESHOLDS_LEARNED = [float(v) for v in np.asarray(core.digi.thresholds)]
        _mv = np.asarray(THRESHOLDS_LEARNED) - np.asarray(THRESHOLDS)
        print(f"  ADC frozen (via {_how}). learned T (e-): {np.round(THRESHOLDS_LEARNED, 1)}")
        print(f"  moved from init (e-)      : {np.round(_mv, 1)}")
        if N_BITS == 2:
            _vp = np.asarray(THRESHOLDS_LEARNED) - np.asarray(PAPER_THRESHOLDS_2BIT)
            print(f"  vs published (e-)         : {np.round(_vp, 1)}")
        assert np.asarray(THRESHOLDS_LEARNED)[0] >= T_MIN - 1e-3, "T0 fell below T_min"

# means must be alive before phase 2
mu_v = np.asarray(core(np.asarray(vg[0][0]), training=False)[0]); yv = np.asarray(vg[0][1])
for nm, i in [("x", 0), ("y", 1)]:
    r = np.corrcoef(mu_v[:, i], yv[:, i])[0, 1]
    print(f"phase-1 {nm}: corr(pred, true) = {r:.4f}")
    assert r > 0.5, f"{nm} means still dead after phase 1 -- STOP"

# phase 2: joint refine under the ORIGINAL loss at lower LR (means settle with cov present)
model.compile(optimizer=tf.keras.optimizers.Adam(LR / 3, clipnorm=1.0), loss=custom_loss)
t0 = time.time()
h2 = model.fit(tg, validation_data=vg, epochs=EPOCHS_P2, shuffle=False, verbose=1,
               callbacks=[tf.keras.callbacks.CSVLogger(os.path.join(OUT_DIR, "history_p2.csv")),
                          NaNStop(),
                          tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=50,
                                                           restore_best_weights=True, verbose=1)])
print("phase 2 done in %.0fs" % (time.time() - t0))

In [ ]:
# ---- 8. train phase 3: error head ONLY, corrected & stable NLL ----
# WHY: loss.custom_loss uses K.sum (batch-size-dependent LR) + a relu diagonal + a clip floor
# that zeros gradients on floored events -> sigma never trains (sigma_x floors to ~0). Here we
# freeze the means (physics ansatz) and train only the error MLP under mean(-log_prob) with a
# SOFTPLUS diagonal (smooth, never hard-zeros the gradient). The dump in cell 10 uses the SAME
# softplus decode, so pulls land at ~1.
def custom_loss_fixed(y, p):
    mu   = p[:, 0:8:2]
    Mdia = 1e-9 + tf.nn.softplus(p[:, 1:8:2])
    Mcov = p[:, 8:]
    z = tf.zeros_like(Mdia[:, 0])
    L = tf.transpose(tf.stack([
        tf.stack([Mdia[:, 0], z,          z,          z         ]),
        tf.stack([Mcov[:, 0], Mdia[:, 1], z,          z         ]),
        tf.stack([Mcov[:, 1], Mcov[:, 2], Mdia[:, 2], z         ]),
        tf.stack([Mcov[:, 3], Mcov[:, 4], Mcov[:, 5], Mdia[:, 3]]),
    ]), perm=[2, 0, 1])
    dist = tfp.distributions.MultivariateNormalTriL(loc=mu, scale_tril=L)
    return tf.reduce_mean(-dist.log_prob(y))

# freeze the physics ansatz (means); only the error MLP stays trainable
for e in core.experts:
    e.get_layer("physics_ansatz").trainable = False
# The ADC was frozen at the end of phase 1. Confirm it: a quantizer still moving
# here would be calibrating the error head against a shifting input distribution.
if N_BITS is not None and TRAINABLE_THR:
    _thr_var = core.digi.quantizer.threshold_deltas_raw
    assert not any(w is _thr_var for w in model.trainable_variables), \
        "ADC thresholds still in model.trainable_variables in phase 3 -- rerun cell 7"
    print("ADC frozen, thresholds (e-):", np.round(np.asarray(core.digi.thresholds), 1))
trainable = [v.name for v in model.trainable_variables]
print("trainable now (expect only error_mlp / chol_entries / hidden):")
for n in trainable: print("  ", n)

model.compile(optimizer=tf.keras.optimizers.Adam(3e-4, clipnorm=1.0), loss=custom_loss_fixed)
t0 = time.time()
h3 = model.fit(tg, validation_data=vg, epochs=EPOCHS_P3, shuffle=False, verbose=1,
               callbacks=[tf.keras.callbacks.CSVLogger(os.path.join(OUT_DIR, "history_p3.csv")),
                          NaNStop(),
                          tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=25,
                                                           restore_best_weights=True, verbose=1)])
print("phase 3 done in %.0fs (val_loss should be NEGATIVE)" % (time.time() - t0))
model.save_weights(os.path.join(OUT_DIR, f"symbolic_n{N_EXPERTS}.weights.h5"))
print("saved weights ->", os.path.join(OUT_DIR, f"symbolic_n{N_EXPERTS}.weights.h5"))

In [ ]:
# ---- 9. post-train metrics: I68, Gaussian-fit pulls, sign accuracy ----
# I68 and the pull fit are byte-for-byte the definitions used in
# compare_baselines_vs_symbolic.ipynb, so the numbers are directly comparable.
from scipy.optimize import curve_fit

def i68(r):
    """paper metric: half-width of the minimal interval containing 68% of residuals"""
    r = np.sort(np.asarray(r)[np.isfinite(r)])
    n = len(r); k = int(np.ceil(0.68 * n))
    if k >= n: return (r[-1] - r[0]) / 2.0
    return (r[k:] - r[:n - k]).min() / 2.0

def _gauss(x, A, mu, sigma):
    return A * np.exp(-(x - mu) ** 2 / (2 * sigma ** 2))

def pull_fit(p):
    h = np.histogram(p, bins=np.linspace(-5, 5, 100))
    xd = h[1][:-1] + np.diff(h[1]) / 2
    try:
        pars, _ = curve_fit(_gauss, xd, h[0], p0=[h[0].max(), 0, 1], maxfev=5000)
        return float(pars[1]), float(abs(pars[2]))
    except Exception:
        return float("nan"), float("nan")

def inverse_cot(c): return np.arctan2(1.0, np.asarray(c, float))

softplus_np = lambda z: np.log1p(np.exp(-np.abs(z))) + np.maximum(z, 0.0)

ls = np.asarray(labels_scale, dtype=float)
P, Yt, S = [], [], []
for i in range(len(EVAL_GEN)):
    cb, lb = EVAL_GEN[i]
    s14 = model(cb, training=False).numpy()
    P.append(s14[:, [0, 2, 4, 6]])
    S.append(softplus_np(s14[:, [1, 3, 5, 7]]))   # same decode as custom_loss_fixed
    Yt.append(np.asarray(lb))
P, S, Yt = np.concatenate(P), np.concatenate(S), np.concatenate(Yt)
print(f"evaluated {len(P)} events at test sigma_noise = {EVAL_NOISE} e-\n")

METRICS = {}
print(f"{'var':<6}{'unit':>5}{'mean':>10}{'I68':>10}{'pull mu':>10}{'pull sig':>10}")
for nm, i, unit in [("x", 0, "um"), ("y", 1, "um"), ("cotA", 2, "deg"), ("cotB", 3, "deg")]:
    if unit == "um":
        t, p, sg = Yt[:, i] * ls[i], P[:, i] * ls[i], S[:, i] * ls[i]
    else:
        t = np.degrees(inverse_cot(Yt[:, i] * ls[i]))
        p = np.degrees(inverse_cot(P[:, i] * ls[i]))
        pu = np.degrees(inverse_cot((P[:, i] + S[:, i]) * ls[i]))
        pd_ = np.degrees(inverse_cot((P[:, i] - S[:, i]) * ls[i]))
        sg = 0.5 * (np.abs(pu - p) + np.abs(pd_ - p))
    r = t - p
    pm, ps = pull_fit(r / (sg + 1e-12))
    METRICS[nm] = dict(unit=unit, mean=float(r.mean()), std=float(r.std()),
                       I68=float(i68(r)), pull_mu=pm, pull_sigma=ps)
    print(f"{nm:<6}{unit:>5}{r.mean():10.4f}{i68(r):10.4f}{pm:10.3f}{ps:10.3f}")

print()
for nm, i in [("cotA", 2), ("cotB", 3)]:
    ti, pr = np.sign(Yt[:, i]), np.sign(P[:, i])
    conf = np.abs(Yt[:, i]) > np.quantile(np.abs(Yt[:, i]), 0.5)
    acc, acc_c = float((pr == ti).mean()), float((pr[conf] == ti[conf]).mean())
    METRICS[nm]["sign_acc"] = acc
    METRICS[nm]["sign_acc_confident_half"] = acc_c
    print(f"{nm} sign accuracy: {acc:.4f}  (confident half: {acc_c:.4f})")

# The sign gate reads Tx, Ty -- a small differential quantity, and the most
# noise-fragile part of the model. Expect this to move before the resolutions do.
if N_BITS is not None:
    _t = [float(v) for v in np.asarray(core.digi.thresholds)]
    print(f"\nthresholds now (e-): {np.round(_t, 1)}"
          f"  moved {np.round(np.asarray(_t) - np.asarray(THRESHOLDS), 1)} e- from init")


In [ ]:
# ---- 10. calibrated parquet dump  (softplus decode -- MUST match custom_loss_fixed) ----
softplus = lambda z: np.log1p(np.exp(-np.abs(z))) + np.maximum(z, 0.0)   # stable, == tf.nn.softplus

P, Yt, S = [], [], []
for i in range(len(EVAL_GEN)):
    cb, lb = EVAL_GEN[i]
    s14 = model(cb, training=False).numpy()
    P.append(s14[:, [0, 2, 4, 6]]); Yt.append(np.asarray(lb)); S.append(s14)
P = np.concatenate(P); Yt = np.concatenate(Yt); S = np.concatenate(S)

# marginal sigma_i = || row_i(L) ||  with L lower-tri, Sigma = L L^T, diagonal = softplus(raw)
Mdia = 1e-9 + softplus(S[:, 1:8:2])
Mcov = S[:, 8:]                        # M21, M31, M32, M41, M42, M43
sig = np.stack([
    np.abs(Mdia[:, 0]),
    np.sqrt(Mcov[:, 0]**2 + Mdia[:, 1]**2),
    np.sqrt(Mcov[:, 1]**2 + Mcov[:, 2]**2 + Mdia[:, 2]**2),
    np.sqrt(Mcov[:, 3]**2 + Mcov[:, 4]**2 + Mcov[:, 5]**2 + Mdia[:, 3]**2),
], axis=1)

df = pd.DataFrame(P, columns=["x", "y", "cotA", "cotB"])
for i, c in enumerate(["xtrue", "ytrue", "cotAtrue", "cotBtrue"]): df[c] = Yt[:, i]
for i, c in enumerate(["sigmax", "sigmay", "sigmacotA", "sigmacotB"]): df[c] = sig[:, i]

pq = os.path.join(OUT_DIR, f"symbolic_n{N_EXPERTS}_vars.parquet")
df.to_parquet(pq)
json.dump({"labels_scale": ls.tolist()}, open(os.path.join(OUT_DIR, "labels_scale.json"), "w"))

print("pull check (want ~1.0 on all four):")
for i, v in enumerate(["x", "y", "cotA", "cotB"]):
    r = df[v + "true"].values - df[v].values
    print(f"  {v:<5} pull_std={(r/sig[:, i]).std():.3f}  sig_med={np.median(sig[:, i]):.4f}")
print("wrote", pq, "|", len(df), "rows")

In [ ]:
# ---- 11. expert scalars + router usage ----
def ansatz_of(e):
    # getattr keeps this working with or without an outer wrapper around the expert
    inner = getattr(e, "inner", e)
    return inner.get_layer("physics_ansatz")

def scalar(ans, key):
    # NOT trainable_weights: phase 3 freezes the ansatz, so that list is empty here
    for w in ans.weights:
        if key in w.name:
            return np.asarray(w.numpy()).ravel()
    return None

expert_scalars = {}
for k, e in enumerate(core.experts):
    ans = ansatz_of(e)
    sc = {w.name: np.asarray(w.numpy()).tolist() for w in ans.weights}
    expert_scalars[f"expert_{k}"] = sc
    def g(key):
        v = scalar(ans, key)
        return None if v is None else np.round(v, 4)
    print(f"expert_{k}: theta_L={g('theta_L')} lorentz_scale={g('lorentz_scale')} "
          f"kA={g('sign_k_alpha')} a0A={g('sign_a0_alpha')} kB={g('sign_k_beta')} a0B={g('sign_a0_beta')}")
json.dump(expert_scalars, open(os.path.join(OUT_DIR, "expert_scalars.json"), "w"), indent=1, default=float)

# physics cross-check: does a0_beta track T*tan(theta_L)?  (skip cleanly if theta_L absent)
for k in range(N_EXPERTS):
    th = scalar(ansatz_of(core.experts[k]), "theta_L")
    if th is None:
        print(f"expert_{k}: theta_L not found -- skipping Lorentz cross-check"); continue
    print(f"expert_{k}: T*tan(theta_L) = {100.0*np.tan(th).item():+.2f} um  (compare a0_beta)")

usage, ent = None, None
if N_EXPERTS > 1:
    Wv = np.concatenate([core.route(vg[i][0]).numpy() for i in range(len(vg))])
    usage = Wv.mean(0)
    ent   = -(Wv * np.log(Wv + 1e-12)).sum(1).mean()
    print("\nrouter usage:", usage.round(3), "| mean entropy: %.3f (max %.3f)" % (ent, np.log(N_EXPERTS)))

In [ ]:
# ---- 12. summary.json -- full provenance, so no run is ever ambiguous again ----
summary = {
    "training": "pure regression (NO teacher/KD) | phase1 MSE means -> phase2 orig-NLL -> phase3 corrected-NLL error head",
    "sign_source": "in-ansatz TimeGrad: sign=tanh(k*(a0-drift)); alpha<-Tx, beta<-Ty (uncrossed, v3 axes)",
    "axis_fix": "in-ansatz v3: cols=x@50um, rows=y@12.5um (measured); fixes pitches, widths and sign observables too",
    "sigma_decode": "softplus diagonal, sigma_i = ||row_i(L)||, Sigma = L L^T (matches custom_loss_fixed)",
    "sign_init": SIGN_INIT,
    "variant": VARIANT, "n_experts": N_EXPERTS,

    # everything about the quantizer, as-built and as-trained
    "digitization": (core.digi.describe(provenance=THR_PROV) if N_BITS is not None
                     else dict(n_bits=None, mode="analog passthrough")),
    "level_mode_requested": (None if N_BITS is None else LEVEL_MODE),
    "level_mode_is_paper_equivalent": (None if N_BITS is None else LEVEL_MODE == "code"),
    "k_schedule": (dict(k_init=K_INIT, k_max=K_MAX, schedule="cosine",
                        annealed=bool(TRAINABLE_THR)) if N_BITS is not None else None),
    "thresholds_learned_e": THRESHOLDS_LEARNED,
    "threshold_trace": (ANNEAL_CB.history if (N_BITS is not None and TRAINABLE_THR
                                              and ANNEAL_CB is not None) else None),

    # noise arm
    "noise": dict(mu=NOISE_MU, train_sigma_e=NOISE_SIGMA, test_sigma_e=EVAL_NOISE,
                  train_test_mismatch=bool(TEST_NOISE_SIGMA is not None),
                  order="noise added to analog charge in the generator, then DigitizeLayer",
                  sigma_noise_nominal_e=SIGMA_NOISE_E),

    "metrics": METRICS,
    "router_hidden": list(ROUTER_HIDDEN), "router_temp": ROUTER_TEMP,
    "total_params": int(model.count_params()),
    "per_expert_params": int(core.experts[0].count_params()),
    "router_params": int(n_router),
    "epochs_p1": len(h1.history["loss"]),
    "epochs_p2": len(h2.history["loss"]),
    "epochs_p3": len(h3.history["loss"]),
    "best_val_loss_p3": float(min(h3.history.get("val_loss", [float("inf")]))),
    "router_usage_val": (usage.tolist() if usage is not None else None),
    "router_entropy_val": (float(ent) if ent is not None else None),
}
json.dump(summary, open(os.path.join(OUT_DIR, "summary.json"), "w"), indent=1, default=float)
print(json.dumps({k: v for k, v in summary.items() if k != "threshold_trace"},
                 indent=1, default=float))


In [ ]:
1